# Scraper — Portal de Subastas BOE
### Paso 1: Conexión y verificación de resultados
---

## Instalación de paquetes

In [15]:
# Ejecuta esta celda solo la primera vez
# Si usas uv: abre la terminal y ejecuta:  uv add cloudscraper beautifulsoup4 lxml
# Si usas pip dentro del notebook quita el comentario de la siguiente linea:
#!pip install cloudscraper beautifulsoup4 lxml

## Imports

In [16]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import time
import pandas as pd

## Crear la sesión cloudscraper

In [17]:
BASE_URL   = 'https://subastas.boe.es'
SEARCH_URL = f'{BASE_URL}/subastas_ava.php'
PAUSA_SEG = 1.5

# Creamos una sesión que simula Chrome en Windows
# cloudscraper resuelve automáticamente los challenges de Cloudflare
scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False}
)

# Cabeceras adicionales para simular un navegador real
scraper.headers.update({
    'Accept-Language' : 'es-ES,es;q=0.9',
    'Accept'          : 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Referer'         : SEARCH_URL,
    'Origin'          : BASE_URL,
})

print('✅ Sesión cloudscraper creada')
print(f'   User-Agent: {scraper.headers["User-Agent"]}')

✅ Sesión cloudscraper creada
   User-Agent: Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36 OPR/45.0.2552.898


## Definir los datos del formulario (POST)

***Inspeccionando cada campo que enviare en la busqueda*** <br>
* input type="radio" id="idTipoBienI" value="I" name="dato[3]" onclick="javascript:document.getElementById('idSubtipoBienITodos').checked='checked'" <br>
* input type="radio" id="idSubtipoBienITodos" value name="dato[4]" checked="checked" <br>
* input type="date" name="dato[17][0]" id="desdeFP"   value  placeholder="dd/mm/yyyy" <br>
* input type="date" name="dato[17][1]" id="hastaFP" value placeholder="dd/mm/yyyy" <br>
* input type="date" name="dato[18][0]"  id="desdeIP" value placeholder="dd/mm/yyyy" <br>
* input type="date" name="dato[18][1]" id="hastaIP" value placeholder="dd/mm/yyyy" <br>
* input type="submit" name="accion" value="Buscar" title="Buscar" class="boton" <br>
 

Basado en la inspección del HTML, el formulario envía:

| Campo HTML | name | value | Qué hace |
|---|---|---|---|
| Radio Inmuebles | `dato[3]` | `I` | Selecciona tipo bien = Inmuebles |
| Radio subcategoría Todos | `dato[4]` | `` (vacío) | Subcategoría = Todos |
| Fecha fin desde | `dato[17][0]` | `YYYY-MM-DD` | Fecha fin subasta - inicio rango |
| Fecha fin hasta | `dato[17][1]` | `YYYY-MM-DD` | Fecha fin subasta - fin rango |
| Fecha inicio desde | `dato[18][0]` | `YYYY-MM-DD` | Fecha inicio subasta - inicio rango |
| Fecha inicio hasta | `dato[18][1]` | `YYYY-MM-DD` | Fecha inicio subasta - fin rango |
| Botón buscar | `accion` | `Buscar` | Dispara la búsqueda |

> **⚠️ Nota sobre las fechas:** el navegador envía `input type=date` en formato `YYYY-MM-DD`  
> aunque el placeholder diga `dd/mm/yyyy`. Eso es solo visual.

In [18]:
# ─── Se puede ajustar las fechas según búsqueda ────────────────────────────
# Formato YYYY-MM-DD (como lo envía el navegador internamente)

FECHA_FIN_DESDE  = '2025-12-01'   # dato[17][0]  → desdeFP  '2026-03-15'
FECHA_FIN_HASTA  = '2025-12-31'   # dato[17][1]  → hastaFP  '2026-04-15'
FECHA_INICIO_DESDE = ''           # dato[18][0]  → desdeIP  '2026-03-01'
FECHA_INICIO_HASTA = ''           # dato[18][1]  → hastaIP  '2026-04-01'

# ─── Cuerpo del POST ────────────────────────────────────────────────────────
form_data = {
    # Tipo de bien → Inmuebles
    'dato[3]'     : 'I',
    # Subcategoría → Todos (vacío = todos los inmuebles)
    'dato[4]'     : '',
    # Fecha fin subasta (rango)
    'dato[17][0]' : FECHA_FIN_DESDE,
    'dato[17][1]' : FECHA_FIN_HASTA,
    # Fecha inicio subasta (rango)
    'dato[18][0]' : FECHA_INICIO_DESDE,
    'dato[18][1]' : FECHA_INICIO_HASTA,
    # Botón de envío
    'accion'      : 'Buscar',
}

print('✅ Formulario definido:')
for k, v in form_data.items():
    print(f'   {k:20s} = {repr(v)}')

✅ Formulario definido:
   dato[3]              = 'I'
   dato[4]              = ''
   dato[17][0]          = '2025-12-01'
   dato[17][1]          = '2025-12-31'
   dato[18][0]          = ''
   dato[18][1]          = ''
   accion               = 'Buscar'


## Enviar el POST y verificar la respuesta

In [19]:
print('🔍 Enviando búsqueda al portal del BOE...')

response = scraper.post(SEARCH_URL, data=form_data, timeout=25)

print(f'   HTTP status  : {response.status_code}')
print(f'   URL final    : {response.url}')
print(f'   Tamaño HTML  : {len(response.text):,} caracteres')

# Verificación rápida
if response.status_code == 200:
    print('✅ Conexión exitosa')
else:
    print(f'❌ Error HTTP {response.status_code}')

🔍 Enviando búsqueda al portal del BOE...
   HTTP status  : 200
   URL final    : https://subastas.boe.es/subastas_ava.php
   Tamaño HTML  : 111,005 caracteres
✅ Conexión exitosa


## Parsear HTML y verificar que llegaron resultados

In [20]:
soup = BeautifulSoup(response.text, 'lxml')

# Extraer todo el texto de la página
texto_pagina = soup.get_text(' ', strip=True)

# Buscar el texto de total: "Resultados X a Y de Z"
match = re.search(
    r'Resultados\s+(\d+)\s+a\s+(\d+)\s+de\s+([\d\.]+)',
    texto_pagina,
    re.IGNORECASE
)

if match:
    desde = int(match.group(1))
    hasta = int(match.group(2))
    total_str = match.group(3).replace('.', '')
    total = int(total_str)
    num_paginas = (total // 50) + 1

    print('=' * 45)
    print(f'✅ Resultados {desde} a {hasta} de {total:,}')
    print(f'📄 Páginas estimadas (50/pág): {num_paginas}')
    print('=' * 45)
else:
    print('❌ No se encontró el texto de resultados.')
    print('\n--- Diagnóstico: primeros 2000 caracteres del HTML ---')
    print(texto_pagina[:2000])

✅ Resultados 1 a 50 de 1,537
📄 Páginas estimadas (50/pág): 31


In [21]:
def extraer_id_busqueda(soup):
    """
    Extrae el token base del id_busqueda desde el link de paginación.
    Ejemplo href: subastas_ava.php?accion=Mas&id_busqueda=<TOKEN>,-50-50
    Nos quedamos solo con <TOKEN> (sin el sufijo ,-N-50).
    """
    link_pag2 = soup.find('a', string='2')
    if link_pag2:
        href = link_pag2.get('href', '')
        m = re.search(r'id_busqueda=(.+?),-\d+-\d+', href)
        if m:
            return m.group(1)
    return None

id_busqueda = extraer_id_busqueda(soup)
if id_busqueda:
    print(f'✅ Token extraído: {id_busqueda[:40]}...')
else:
    print('❌ No encontrado')

✅ Token extraído: b1dsZHcyK2F3Z2VwRFVqaXQ1OGVPbGZZWnlDUzY0...


In [22]:
def extraer_listado(soup):
    """
    Extrae datos de cada <li class='resultado-busqueda'>.

    Estructura HTML:
      <li class='resultado-busqueda'>
        <h3>SUBASTA SUB-XX-...</h3>          ← referencia
        <h4>ORGANISMO - CIUDAD</h4>          ← organismo
        <p>Expediente: ...</p>               ← opcional
        <p>Estado: ... [Fecha...]</p>
        <p>Descripción del bien</p>
        <a class='resultado-busqueda-link-otro' href='...'>Más...</a>
      </li>
    """
    registros = []
    items = soup.find_all('li', class_='resultado-busqueda')

    for item in items:
        reg = {}

        # Referencia (h3) — puede venir con "(2 lotes)"
        h3 = item.find('h3')
        if h3:
            texto_h3 = h3.get_text(strip=True)
            m_ref = re.search(r'(SUB-[A-Z0-9\-]+)', texto_h3)
            reg['referencia'] = m_ref.group(1) if m_ref else texto_h3
            m_lotes = re.search(r'\((\d+) lotes?\)', texto_h3, re.I)
            reg['num_lotes_listado'] = int(m_lotes.group(1)) if m_lotes else 1

        # Organismo (h4)
        h4 = item.find('h4')
        reg['organismo'] = h4.get_text(strip=True) if h4 else None

        # Párrafos: expediente, estado, descripción
        reg['expediente']             = None
        reg['estado']                 = None
        reg['fecha_conclusion_listado'] = None
        reg['descripcion']            = None

        for p in item.find_all('p'):
            texto_p = p.get_text(' ', strip=True)
            if not texto_p:
                continue
            if texto_p.startswith('Expediente:'):
                reg['expediente'] = texto_p.replace('Expediente:', '').strip()
            elif texto_p.startswith('Estado:'):
                m_estado = re.match(r'Estado:\s*([^\[\-]+)', texto_p)
                reg['estado'] = m_estado.group(1).strip() if m_estado else texto_p
                m_fecha = re.search(r'Fecha de conclusión:\s*([\d/]+ a las [\d:]+)', texto_p)
                reg['fecha_conclusion_listado'] = m_fecha.group(1) if m_fecha else None
            else:
                reg['descripcion'] = texto_p  # último párrafo no vacío

        # URL del detalle + id_sub limpio
        enlace = item.find('a', class_='resultado-busqueda-link-otro')
        if enlace:
            href = enlace['href'].replace('./', '')
            reg['url_detalle'] = f"{BASE_URL}/{href}"
            m_idsub = re.search(r'idSub=([^&]+)', href)
            reg['id_sub'] = m_idsub.group(1) if m_idsub else None

        registros.append(reg)

    return registros


# Prueba con la primera página
registros_p1 = extraer_listado(soup)
print(f'✅ Tarjetas extraídas en página 1: {len(registros_p1)}')
print('\nEjemplo — primer registro:')
for k, v in registros_p1[0].items():
    print(f'  {k:30s}: {v}')

✅ Tarjetas extraídas en página 1: 50

Ejemplo — primer registro:
  referencia                    : SUB-RC-2025-3800100125018
  num_lotes_listado             : 1
  organismo                     : Agencia Tributaria Canaria - Las Palmas de Gran Canaria (Comunidad Autónoma de Canarias)
  expediente                    : None
  estado                        : Suspendida
  fecha_conclusion_listado      : None
  descripcion                   : URBANA: Número veinticinco. -Unidad de Alojamiento número “setecientos cincuenta y siete”, Bungalow tipo “B”, número “Treinta y cinco” en la precontratación, sito en el Bloque Número V de lo que integran el Complejo denominado Green Sea, construido sobre la parcela B del polígono once de la Urbanización “Las Burras”. superficie edificada conjunta de sesenta y dos metros dos decímetros cuadrados, incluyendo el jardín privado
  url_detalle                   : https://subastas.boe.es/detalleSubasta.php?idSub=SUB-RC-2025-3800100125018&idBus=b1dsZHcyK2F3Z2Vw

In [23]:
def limpiar_importe(texto):
    #'55.110,30 €' → 55110.30  |  'Sin puja mínima' → None
    #Si esta vacio o encuentra la palabra 'sin' o 'ver' (re.I ingnora mayusculas/minusculas)
    if not texto or re.search(r'sin|ver ', texto, re.I): 
        return None
    
    #Reemplazar cualquier caracter que no sea digito ni coma por ' '
    limpio = re.sub(r'[^\d,]', '', texto)   # queda '55110,30'
    if not limpio:
        return None
    return float(limpio.replace(',', '.'))


def extraer_detalle(scraper, url):
    """
    Descarga la ficha individual y extrae la tabla <th>/<td>.

    Campos que extrae:
      tipo_subasta, cuenta_expediente, fecha_inicio, fecha_conclusion,
      cantidad_reclamada_eur, lotes, anuncio_boe,
      valor_subasta_eur, tasacion_eur, puja_minima_eur,
      tramo_pujas_eur, deposito_eur
    """
    try:
        r = scraper.get(url, timeout=20)
        r.raise_for_status()
    except Exception as e:
        print(f'  ⚠️  Error: {e}')
        return {}

    soup_det = BeautifulSoup(r.text, 'lxml')
    datos = {}

    MAPA_CAMPOS = {
        'Identificador'       : 'referencia_det',
        'Tipo de subasta'     : 'tipo_subasta',
        'Cuenta expediente'   : 'cuenta_expediente',
        'Fecha de inicio'     : 'fecha_inicio',
        'Fecha de conclusión' : 'fecha_conclusion',
        'Cantidad reclamada'  : 'cantidad_reclamada_eur',
        'Lotes'               : 'lotes',
        'Forma adjudicación'  : 'forma_adjudicacion',
        'Anuncio BOE'         : 'anuncio_boe',
        'Valor subasta'       : 'valor_subasta_eur',
        'Tasación'            : 'tasacion_eur',
        'Puja mínima'         : 'puja_minima_eur',
        'Tramos entre pujas'  : 'tramo_pujas_eur',
        'Importe del depósito': 'deposito_eur',
    }

    for fila in soup_det.find_all('tr'):
        th = fila.find('th')
        td = fila.find('td')
        if not th or not td:
            continue

        clave = th.get_text(strip=True)
        valor = td.get_text(' ', strip=True)
        col   = MAPA_CAMPOS.get(clave)

        if col is None:
            continue

        if col.endswith('_eur'):
            datos[col] = limpiar_importe(valor)
        elif col.startswith('fecha_'):
            # '11-11-2025 18:00:00 CET (ISO: ...)' → '11-11-2025 18:00:00 CET'
            datos[col] = valor.split('(')[0].strip()
        elif col == 'lotes':
            #Si lotes contienve el texto 'sin' datos[lotes] = 1
            datos[col] = 1 if 'sin' in valor.lower() else int(re.search(r'\d+', valor).group())
        else:
            datos[col] = valor

    return datos


# Prueba con la 4ª subasta (vivienda de Madrid que viste en el inspector)
url_prueba = registros_p1[3]['url_detalle']
print(f'🔍 Probando detalle...')
detalle_prueba = extraer_detalle(scraper, url_prueba)
print('✅ Datos del detalle:')
for k, v in detalle_prueba.items():
    print(f'  {k:25s}: {v}')

🔍 Probando detalle...
✅ Datos del detalle:
  referencia_det           : SUB-AT-2025-25R0886001399
  tipo_subasta             : AGENCIA TRIBUTARIA
  fecha_inicio             : 11-11-2025 18:00:00 CET
  fecha_conclusion         : 01-12-2025 18:00:00 CET
  lotes                    : 1
  anuncio_boe              : BOE-B-2025-41050
  valor_subasta_eur        : 49200.0
  tasacion_eur             : 82000.0
  puja_minima_eur          : 4920.0
  tramo_pujas_eur          : 1000.0
  deposito_eur             : 2460.0


In [24]:
def get_pagina_listado(scraper, n_pagina, id_busqueda):
    """
    Paginación real del BOE:
      página 1 → ,-0-50
      página 2 → ,-50-50
      página 3 → ,-100-50
      página N → ,-(N-1)*50-50
    """
    offset = (n_pagina - 1) * 50
    id_busqueda_completo = f'{id_busqueda},-{offset}-50'

    url = f'{SEARCH_URL}?accion=Mas&id_busqueda={id_busqueda_completo}'

    try:
        r = scraper.get(url, timeout=20)
        r.raise_for_status()
        return BeautifulSoup(r.text, 'lxml')
    except Exception as e:
        print(f'  ⚠️  Error en página {n_pagina}: {e}')
        return None


# Prueba: página 2
soup_p2 = get_pagina_listado(scraper, n_pagina=2, id_busqueda=id_busqueda)
if soup_p2:
    regs_p2 = extraer_listado(soup_p2)
    print(f'✅ Página 2: {len(regs_p2)} tarjetas')
    print(f'   Primera: {regs_p2[0]["referencia"]}')
    print(f'   Última:  {regs_p2[-1]["referencia"]}')

print()

✅ Página 2: 50 tarjetas
   Primera: SUB-JA-2025-253169
   Última:  SUB-JA-2025-253971



In [25]:
MAX_PAGINAS = num_paginas   # ← sube a num_paginas cuando valides que funciona

todos_registros = []

for n_pag in range(1, MAX_PAGINAS + 1):
    print(f'\n📄 Página {n_pag}/{MAX_PAGINAS}')

    if n_pag == 1:
        soup_pag = soup  # ya descargada
    else:
        posicion = (n_pag - 1) * 50 + 1
        soup_pag = get_pagina_listado(scraper, n_pagina=n_pag, id_busqueda=id_busqueda)
        if soup_pag is None:
            print(f'  ⚠️  Sin respuesta, saltando')
            continue
        time.sleep(PAUSA_SEG)

    registros_listado = extraer_listado(soup_pag)
    print(f'  Tarjetas: {len(registros_listado)}')

    for i, reg in enumerate(registros_listado):
        url = reg.get('url_detalle')
        if url:
            detalle = extraer_detalle(scraper, url)
            reg.update(detalle)
            time.sleep(PAUSA_SEG)
        todos_registros.append(reg)
        if (i + 1) % 10 == 0:
            print(f'    → {i+1}/{len(registros_listado)} procesados')

print(f'\n✅ Total registros: {len(todos_registros)}')


📄 Página 1/31
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

📄 Página 2/31
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

📄 Página 3/31
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

📄 Página 4/31
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

📄 Página 5/31
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

📄 Página 6/31
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50/50 procesados

📄 Página 7/31
  Tarjetas: 50
    → 10/50 procesados
    → 20/50 procesados
    → 30/50 procesados
    → 40/50 procesados
    → 50

In [27]:
df = pd.DataFrame(todos_registros).copy()
df

,referencia,num_lotes_listado,organismo,expediente,estado,fecha_conclusion_listado,descripcion,url_detalle,id_sub,referencia_det,...,anuncio_boe,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur,fecha_inicio,fecha_conclusion,cuenta_expediente,forma_adjudicacion
0,SUB-RC-2025-3800100125018,1,Agencia Tributaria Canaria - Las Palmas de Gra...,NaN,Suspendida,NaN,URBANA: Número veinticinco. -Unidad de Alojami...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-RC-2025-3800100125018,SUB-RC-2025-3800100125018,...,BOE-B-2025-39962,105744.66,105744.66,NaN,1500.00,5287.23,NaN,NaN,NaN,NaN
1,SUB-AT-2025-24R4686001676,1,U.R. SUBASTAS VALENCIA - VALENCIA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,01/12/2025 a las 18:00:00,"VIVIENDA. CL/ SIENA, 37 BJ J. 28027 - MADRID (...",https://subastas.boe.es/detalleSubasta.php?idS...,SUB-AT-2025-24R4686001676,SUB-AT-2025-24R4686001676,...,BOE-B-2025-41064,55110.30,311336.41,NaN,1000.00,2755.51,11-11-2025 18:00:00 CET,01-12-2025 18:00:00 CET,NaN,NaN
2,SUB-AT-2025-25R0886001398,1,U.R. SUBASTAS CATALUÑA - BARCELONA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,01/12/2025 a las 18:00:00,SOLAR. UR FONT DEL BOSC 108 MEDIONA BARCELONA,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-AT-2025-25R0886001398,SUB-AT-2025-25R0886001398,...,BOE-B-2025-41049,47400.00,79000.00,4740.00,1000.00,2370.00,11-11-2025 18:00:00 CET,01-12-2025 18:00:00 CET,NaN,NaN
3,SUB-AT-2025-25R0886001399,1,U.R. SUBASTAS CATALUÑA - BARCELONA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,01/12/2025 a las 18:00:00,SOLAR. UR FONT DEL BOSC . C/ BELLOC 131 MEDION...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-AT-2025-25R0886001399,SUB-AT-2025-25R0886001399,...,BOE-B-2025-41050,49200.00,82000.00,4920.00,1000.00,2460.00,11-11-2025 18:00:00 CET,01-12-2025 18:00:00 CET,NaN,NaN
4,SUB-AT-2025-25R0886001455,1,U.R. SUBASTAS CATALUÑA - BARCELONA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,01/12/2025 a las 18:00:00,100% PLENO DOMINIO DE GARAJE. PD/ DEL CAMP . 2...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-AT-2025-25R0886001455,SUB-AT-2025-25R0886001455,...,BOE-B-2025-41051,15264.48,15264.48,1526.45,500.00,763.22,11-11-2025 18:00:00 CET,01-12-2025 18:00:00 CET,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1532,SUB-JA-2025-253760,1,JUZGADO 1 INSTANCIA 8 - PALMA DE MALLORCA,0052/15,Pendiente de finalización y devolución de depó...,30/12/2025 a las 03:17:25,"URBANA NUMERO DOS DE ORDEN. VIVIENDA A, o dere...",https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-253760,SUB-JA-2025-253760,...,BOE-B-2025-42325,244500.00,244500.00,NaN,4890.00,12225.00,21-11-2025 18:00:00 CET,30-12-2025 03:17:25 CET,0470 0000 06 0052 15,NaN
1533,SUB-JA-2025-255033,1,JUZGADO 1 INST E INSTRUCC. 2 - VINAROS,0044/93,Pendiente de finalización y devolución de depó...,30/12/2025 a las 04:31:37,:Usufructo de la Finca registral núm. 39944.\n...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-255033,SUB-JA-2025-255033,...,BOE-B-2025-45287,8163.71,0.00,NaN,163.27,408.18,09-12-2025 18:00:00 CET,30-12-2025 04:31:37 CET,1354 0000 05 0044 93,NaN
1534,SUB-JA-2025-254936,1,JUZGADO 1 INSTANCIA 2 - ALMERIA,2217/22,Pendiente de finalización y devolución de depó...,30/12/2025 a las 06:06:40,Identificación de la finca: URBANA: SOLAR EN L...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-254936,SUB-JA-2025-254936,...,BOE-B-2025-45290,51011.73,0.00,NaN,1020.23,2550.58,09-12-2025 18:00:00 CET,30-12-2025 06:06:40 CET,0224 0000 06 2217 22,NaN
1535,SUB-RC-2025-0026I20250168,1,Suma Gestión Tributaria. Diputación de Alicant...,NaN,Finalizada y depósitos con reserva devueltos,30/12/2025 a las 06:11:02,NaN,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-RC-2025-0026I20250168,SUB-RC-2025-0026I20250168,...,BOE-B-2025-45382,110556.00,110556.00,55278.00,3000.00,5527.80,09-12-2025 18:00:00 CET,30-12-2025 06:11:02 CET,NaN,NaN

In [28]:
df = pd.DataFrame(todos_registros).copy()

# Convertir fechas
for col in ['fecha_inicio', 'fecha_conclusion']:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col].str.extract(r'(\d{2}-\d{2}-\d{4})')[0],
            format='%d-%m-%Y',
            errors='coerce'
        )
   
print(f'Shape: {df.shape}')
print(f'Columnas: {list(df.columns)}')
df.head(3)

Shape: (1537, 23)
Columnas: ['referencia', 'num_lotes_listado', 'organismo', 'expediente', 'estado', 'fecha_conclusion_listado', 'descripcion', 'url_detalle', 'id_sub', 'referencia_det', 'tipo_subasta', 'cantidad_reclamada_eur', 'lotes', 'anuncio_boe', 'valor_subasta_eur', 'tasacion_eur', 'puja_minima_eur', 'tramo_pujas_eur', 'deposito_eur', 'fecha_inicio', 'fecha_conclusion', 'cuenta_expediente', 'forma_adjudicacion']


,referencia,num_lotes_listado,organismo,expediente,estado,fecha_conclusion_listado,descripcion,url_detalle,id_sub,referencia_det,...,anuncio_boe,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur,fecha_inicio,fecha_conclusion,cuenta_expediente,forma_adjudicacion
0,SUB-RC-2025-3800100125018,1,Agencia Tributaria Canaria - Las Palmas de Gra...,NaN,Suspendida,NaN,URBANA: Número veinticinco. -Unidad de Alojami...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-RC-2025-3800100125018,SUB-RC-2025-3800100125018,...,BOE-B-2025-39962,105744.66,105744.66,NaN,1500.0,5287.23,NaT,NaT,NaN,NaN
1,SUB-AT-2025-24R4686001676,1,U.R. SUBASTAS VALENCIA - VALENCIA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,01/12/2025 a las 18:00:00,"VIVIENDA. CL/ SIENA, 37 BJ J. 28027 - MADRID (...",https://subastas.boe.es/detalleSubasta.php?idS...,SUB-AT-2025-24R4686001676,SUB-AT-2025-24R4686001676,...,BOE-B-2025-41064,55110.30,311336.41,NaN,1000.0,2755.51,2025-11-11,2025-12-01,NaN,NaN
2,SUB-AT-2025-25R0886001398,1,U.R. SUBASTAS CATALUÑA - BARCELONA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,01/12/2025 a las 18:00:00,SOLAR. UR FONT DEL BOSC 108 MEDIONA BARCELONA,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-AT-2025-25R0886001398,SUB-AT-2025-25R0886001398,...,BOE-B-2025-41049,47400.00,79000.00,4740.0,1000.0,2370.00,2025-11-11,2025-12-01,NaN,NaN


In [30]:
df.to_csv('../data/subastas_inmuebles_dic2025.csv', index=False, encoding='utf-8-sig') #subastas_inmuebles_dic2025.csv
print(f'✅ Guardado: {len(df)} filas, {df.shape[1]} columnas')
print()
print(df.dtypes)

✅ Guardado: 1537 filas, 23 columnas

referencia                             str
num_lotes_listado                    int64
organismo                              str
expediente                             str
estado                                 str
fecha_conclusion_listado               str
descripcion                            str
url_detalle                            str
id_sub                                 str
referencia_det                         str
tipo_subasta                           str
cantidad_reclamada_eur             float64
lotes                                int64
anuncio_boe                            str
valor_subasta_eur                  float64
tasacion_eur                       float64
puja_minima_eur                    float64
tramo_pujas_eur                    float64
deposito_eur                       float64
fecha_inicio                datetime64[us]
fecha_conclusion            datetime64[us]
cuenta_expediente                      str
forma_adjudicacio